# 🇵🇪 PeruRoute AI — Pipeline Generativo Multimodal para Turismo Personalizado

**Diploma AI Engineer · Módulo: Ingeniería de Aplicaciones Generativas**

---

## ¿Qué hace este notebook?

Implementa un pipeline generativo completo en dos módulos integrados:

```
MÓDULO 1 — Fine-tuning LLM
  Perfil del turista → Qwen2.5-3B + Unsloth + QLoRA → Itinerario personalizado

MÓDULO 2 — Fine-tuning de imagen
  Logo PeruRoute AI + Trigger word → SDXL + DreamBooth LoRA → Logo en productos

MÓDULO 3 — Pipeline integrado
  Perfil del turista → Itinerario (LLM) + Imagen del destino (SDXL) → Output completo
```

## Requisitos
- ✅ GPU T4 de Google Colab (Entorno de ejecución → Cambiar tipo → T4 GPU)
- ✅ Cuenta HuggingFace gratuita con token
- ✅ Google Drive con las imágenes del logo en `/brand_mockup_training/imagenes/`

## Tiempo estimado en T4
| Módulo | Tiempo |
|---|---|
| Instalación | ~10 min |
| Fine-tuning LLM (Parte 1) | ~30-40 min |
| Fine-tuning imagen (Parte 2) | ~25-35 min |
| Pipeline integrado (Parte 3) | ~5 min |
| **Total** | **~70-90 min** |

> 💡 Ejecuta las celdas **secuencialmente**. No saltes secciones.

---
# ⚙️ CELDA 0 — Verificar GPU y conectar Drive

In [ ]:
# ============================================================
# CELDA 0: Verificar GPU en Kaggle
# ============================================================
import subprocess, torch

# Verificar GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detectada:')
    for line in result.stdout.split('\n')[:15]:
        print(line)
else:
    print('❌ No se detectó GPU.')
    print('👉 Ve a: Settings → Accelerator → GPU T4 x2')
    raise SystemExit('Activa la GPU primero.')

print(f'\n✅ PyTorch version : {torch.__version__}')
print(f'   CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU             : {torch.cuda.get_device_name(0)}')
    print(f'   CUDA version    : {torch.version.cuda}')
print('\n✅ Entorno Kaggle listo.')

---
# 📦 CELDA 1 — Instalar dependencias

> ⚠️ Después de ejecutar esta celda, **reinicia la sesión** y vuelve a ejecutar desde la Celda 0.

In [ ]:
# ============================================================
# CELDA 1: Instalar todas las dependencias del proyecto
# ============================================================
print('📦 Instalando dependencias del Módulo 1 (LLM)...')
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

print('\n📦 Instalando dependencias del Módulo 2 (Imágenes)...')
!pip install -q --upgrade \
    huggingface_hub \
    git+https://github.com/huggingface/diffusers.git \
    transformers \
    safetensors \
    torchvision \
    datasets

print('\n✅ Instalación completa.')
print('👉 REINICIA LA SESIÓN: Menú → Entorno de ejecución → Reiniciar sesión')
print('   Luego ejecuta desde la Celda 0.')

---
# 🔑 CELDA 2 — Configuración global del proyecto

> 📝 **Edita aquí** tu token de HuggingFace y los parámetros de tu marca.

In [30]:
# ============================================================
# CELDA 2: Configuración global — EDITA AQUÍ
# ============================================================
import os
print("=== Estructura /kaggle/input ===")
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(f"📁 {dirname}")
    for f in filenames[:3]:
        print(f"   {f}")

from huggingface_hub import login
import os

# ── HuggingFace ───────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")  # Configura en Kaggle Secrets 🔑
                                            # https://huggingface.co/settings/tokens

# ── Módulo 1: LLM ────────────────────────────────────────
LLM_MODEL_NAME   = "unsloth/Qwen2.5-3B-Instruct"
LLM_MAX_SEQ_LEN  = 2048
LLM_LORA_R       = 16
LLM_LORA_ALPHA   = 16
LLM_EPOCHS       = 3
LLM_BATCH_SIZE   = 2
LLM_GRAD_ACCUM   = 4
LLM_LR           = 2e-4
LLM_OUTPUT_DIR   = "/kaggle/working/peruroute_llm_lora"

# ── Módulo 2: Imagen ─────────────────────────────────────
IMG_TRIGGER_WORD = "PERUROUTE"             # Palabra única de la marca
IMG_BRAND_COLORS = "dark green and gold"   # Paleta
IMG_BRAND_STYLE  = "andean inspired travel brand"
IMG_MODEL_NAME   = "runwayml/stable-diffusion-v1-5"
IMG_LORA_RANK    = 16
IMG_MAX_STEPS    = 200
IMG_LR           = 1e-4
IMG_RESOLUTION   = 512
IMG_IMAGES_DIR   = "/kaggle/input/datasets/roxanasiu/peruroute-logos"
IMG_OUTPUT_DIR   = "/kaggle/working/brand_output"

# ── Directorios de trabajo ───────────────────────────────
DATASET_CSV_PATH = "/kaggle/input/datasets/roxanasiu/peruroute-dataset/dataset_turismo_peru.csv"

os.makedirs(LLM_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMG_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMG_IMAGES_DIR, exist_ok=True)

# Login HuggingFace
if HF_TOKEN == "hf_XXXXXXXXXXXXXXXXXXXXXXXX":
    raise ValueError('❌ Pega tu token de HuggingFace en HF_TOKEN.')
login(token=HF_TOKEN)

print('✅ Configuración lista.')
print(f'   LLM model   : {LLM_MODEL_NAME}')
print(f'   Trigger word: {IMG_TRIGGER_WORD}')
print(f'   Dataset CSV : {DATASET_CSV_PATH}')

=== Estructura /kaggle/input ===
📁 /kaggle/input
📁 /kaggle/input/datasets
📁 /kaggle/input/datasets/roxanasiu
📁 /kaggle/input/datasets/roxanasiu/peruroute-dataset
   dataset_turismo_peru.csv
📁 /kaggle/input/datasets/roxanasiu/peruroute-logos
   03_logo_fondo_negro.png
   12_logo_fondo_verde_medio.png
   10_escala_grises.png
✅ Configuración lista.
   LLM model   : unsloth/Qwen2.5-3B-Instruct
   Trigger word: PERUROUTE
   Dataset CSV : /kaggle/input/datasets/roxanasiu/peruroute-dataset/dataset_turismo_peru.csv


---
# 🤖 PARTE 1 — Fine-tuning del LLM para generación de itinerarios

Entrenamos **Qwen2.5-3B-Instruct** con técnica **QLoRA** via Unsloth sobre un dataset de 800 perfiles de turistas y sus itinerarios personalizados en Perú.

```
Entrada:  perfil del turista (días, presupuesto, intereses, destino, grupo)
    ↓
Fine-tuning: formato Alpaca (instruction / input / output)
    ↓
Salida:   itinerario día a día con tips de transporte, alojamiento y cultura
```

## CELDA 1.1 — Cargar modelo base con Unsloth

In [ ]:
# Desinstalar bitsandbytes y usar unsloth sin cuantización 4-bit
!pip uninstall bitsandbytes -y -q
!pip install -q "unsloth[cu121] @ git+https://github.com/unslothai/unsloth.git"

import os
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["BNB_CUDA_VERSION"] = "130"

In [ ]:
!pip install -q "transformers==4.51.3"
!pip install -q "trl==0.8.6"

# Reiniciar kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [2]:
# ============================================================
# CELDA 1.1: Cargar Qwen2.5-3B con QLoRA 4-bit
# ============================================================
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = LLM_MODEL_NAME,
    max_seq_length= LLM_MAX_SEQ_LEN,
    dtype         = None,       # Auto: Float16 en T4, BFloat16 en Ampere+
    load_in_4bit  = False,       # QLoRA: cuantización 4-bit
)

print(f'✅ Modelo cargado: {LLM_MODEL_NAME}')
print(f'   Parámetros totales: {model.num_parameters()/1e9:.2f}B')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/__init__.py:404: UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4.56.0.
  _install_fused_forward()
W0602 23:28:49.227000 1011 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-06-02 23:28:52.733146: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780442932.980036    1011 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780442933.047831    1011 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has alre

🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Modelo cargado: unsloth/Qwen2.5-3B-Instruct
   Parámetros totales: 3.09B


## CELDA 1.2 — Configurar adaptadores LoRA

In [5]:
# ============================================================
# CELDA 1.2: Configurar LoRA — solo estos parámetros se entrenan
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r                        = LLM_LORA_R,
    target_modules           = ["q_proj", "k_proj", "v_proj", "o_proj",
                                 "gate_proj", "up_proj", "down_proj"],
    lora_alpha               = LLM_LORA_ALPHA,
    lora_dropout             = 0,
    bias                     = "none",
    use_gradient_checkpointing= "unsloth",
    random_state             = 42,
    use_rslora               = False,
    loftq_config             = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA configurado — r={LLM_LORA_R}, alpha={LLM_LORA_ALPHA}')
print(f'   Parámetros entrenables: {trainable:,} ({100*trainable/total:.2f}% del total)')

Unsloth: Already have LoRA adapters! We shall skip this step.


✅ LoRA configurado — r=16, alpha=16
   Parámetros entrenables: 29,933,568 (0.96% del total)


## CELDA 1.3 — Cargar y preparar el dataset

In [6]:
# ============================================================
# CELDA 1.3: Cargar dataset CSV y formatear en Alpaca
# ============================================================
import pandas as pd
from datasets import Dataset

# Formato Alpaca: el modelo aprende a completar la sección ### Respuesta:
ALPACA_PROMPT = """A continuación hay una instrucción que describe una tarea, \
junto con una entrada que proporciona más contexto. \
Escribe una respuesta que complete apropiadamente la solicitud.

### Instrucción:
{}

### Entrada:
{}

### Respuesta:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    """Convierte cada fila del CSV al formato Alpaca."""
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = ALPACA_PROMPT.format(inst, inp, out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Cargar CSV
df      = pd.read_csv(DATASET_CSV_PATH)
dataset = Dataset.from_pandas(df)
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f'✅ Dataset cargado: {len(dataset)} ejemplos')
print(f'   Columnas: {list(df.columns)}')
print(f'\n--- Ejemplo de prompt (primeros 400 chars) ---')
print(dataset[0]["text"][:400] + '...')

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

✅ Dataset cargado: 800 ejemplos
   Columnas: ['id', 'dias_viaje', 'presupuesto', 'intereses', 'origen_turista', 'idioma', 'grupo', 'region_preferida', 'destino_principal', 'temporada', 'instruction', 'input', 'output']

--- Ejemplo de prompt (primeros 400 chars) ---
A continuación hay una instrucción que describe una tarea, junto con una entrada que proporciona más contexto. Escribe una respuesta que complete apropiadamente la solicitud.

### Instrucción:
Genera un itinerario turístico personalizado para el Perú basado en el perfil del viajero. El itinerario debe incluir actividades por día, recomendaciones de transporte, alojamiento según presupuesto y tips ...


## CELDA 1.4 — Entrenamiento del LLM

In [8]:
import os, subprocess

# Encontrar libnvJitLink en el sistema
result = subprocess.run(['find', '/usr', '-name', 'libnvJitLink*'], 
                       capture_output=True, text=True)
print("Librerías encontradas:")
print(result.stdout)

# Forzar path
paths = [p.rsplit('/',1)[0] for p in result.stdout.strip().split('\n') if p]
for p in paths:
    os.environ["LD_LIBRARY_PATH"] = p + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print(f"✅ Agregado: {p}")

# Reinstalar bitsandbytes apuntando a CUDA correcto
!pip install -q bitsandbytes --extra-index-url https://pypi.nvidia.com

Librerías encontradas:
/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib/libnvJitLink.so.12
/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libnvJitLink.so.13
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libnvJitLink_static.a
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libnvJitLink.so
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libnvJitLink.so.12.8.93
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libnvJitLink.so.12

✅ Agregado: /usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib
✅ Agregado: /usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib
✅ Agregado: /usr/local/cuda-12.8/targets/x86_64-linux/lib
✅ Agregado: /usr/local/cuda-12.8/targets/x86_64-linux/lib
✅ Agregado: /usr/local/cuda-12.8/targets/x86_64-linux/lib
✅ Agregado: /usr/local/cuda-12.8/targets/x86_64-linux/lib


In [9]:
# ============================================================
# CELDA 1.4: Fine-tuning con SFTTrainer
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = dataset,
    dataset_text_field= "text",
    max_seq_length    = LLM_MAX_SEQ_LEN,
    dataset_num_proc  = 2,
    packing           = False,
    args = TrainingArguments(
        per_device_train_batch_size = LLM_BATCH_SIZE,
        gradient_accumulation_steps = LLM_GRAD_ACCUM,
        warmup_steps                = 10,
        num_train_epochs            = LLM_EPOCHS,
        learning_rate               = LLM_LR,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        optim                       = "adamw_torch_fused",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 42,
        output_dir                  = "/kaggle/working/llm_checkpoints",
        report_to                   = "none",
        include_num_input_tokens_seen = False,
    ),
)

# Estadísticas previas
gpu_stats        = torch.cuda.get_device_properties(0)
start_vram       = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
max_vram         = round(gpu_stats.total_memory / 1024**3, 2)
print(f'🚀 Iniciando entrenamiento LLM...')
print(f'   GPU: {gpu_stats.name} | VRAM disponible: {max_vram} GB')
print(f'   Épocas: {LLM_EPOCHS} | Batch efectivo: {LLM_BATCH_SIZE * LLM_GRAD_ACCUM}')

trainer_stats = trainer.train()

used_vram = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f'\n✅ Entrenamiento completado!')
print(f'   Tiempo total : {trainer_stats.metrics["train_runtime"]:.0f} segundos')
print(f'   Loss final   : {trainer_stats.metrics["train_loss"]:.4f}')
print(f'   VRAM peak    : {used_vram} GB ({100*used_vram/max_vram:.1f}%)')

Map (num_proc=2):   0%|          | 0/800 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/unsloth/models/_utils.py:2407: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  _original_trainer_init(self, *args, **kwargs)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 3 | Total steps = 150
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


🚀 Iniciando entrenamiento LLM...
   GPU: Tesla T4 | VRAM disponible: 14.56 GB
   Épocas: 3 | Batch efectivo: 8


<IPython.core.display.HTML object>


✅ Entrenamiento completado!
   Tiempo total : 1416 segundos
   Loss final   : 0.2221
   VRAM peak    : 8.31 GB (57.1%)


## CELDA 1.5 — Guardar modelo fine-tuned

In [10]:
# ============================================================
# CELDA 1.5: Guardar adaptadores LoRA en Drive
# ============================================================
model.save_pretrained(LLM_OUTPUT_DIR)
tokenizer.save_pretrained(LLM_OUTPUT_DIR)

print(f'✅ Modelo LLM fine-tuned guardado en: {LLM_OUTPUT_DIR}')
print(f'   (Solo los adaptadores LoRA — ~20-50 MB)')

✅ Modelo LLM fine-tuned guardado en: /kaggle/working/peruroute_llm_lora
   (Solo los adaptadores LoRA — ~20-50 MB)


## CELDA 1.6 — Comparativa: modelo base vs fine-tuned

In [12]:
# ============================================================
# CELDA 1.6: Comparativa modelo base vs fine-tuned
# ============================================================
FastLanguageModel.for_inference(model)

def generar_itinerario(instruccion, perfil, modelo, tok):
    """Genera un itinerario dado un perfil de turista."""
    prompt = ALPACA_PROMPT.format(instruccion, perfil, "")
    inputs = tok([prompt], return_tensors="pt").to("cuda")
    outputs = modelo.generate(
        **inputs, max_new_tokens=300,
        temperature=0.7, top_p=0.9, use_cache=True
    )
    resultado = tok.decode(outputs[0], skip_special_tokens=True)
    if "### Respuesta:" in resultado:
        return resultado.split("### Respuesta:")[1].strip()
    return resultado

INSTRUCCION = "Genera un itinerario turístico personalizado para el Perú basado en el perfil del viajero."
PERFIL_TEST = "Turista de Argentina, viaja en pareja, 5 días, presupuesto medio, intereses: cultura, región: sierra, destino: Machu Picchu, temporada: seca (mayo-octubre)."

print('=' * 60)
print('PERFIL DE PRUEBA:')
print(PERFIL_TEST)
print('=' * 60)

# Modelo fine-tuned
print('\n✨ MODELO FINE-TUNED (PeruRoute AI):')
print('-' * 40)
resp_ft = generar_itinerario(INSTRUCCION, PERFIL_TEST, model, tokenizer)
print(resp_ft)

# Modelo base
model_base, tok_base = FastLanguageModel.from_pretrained(
    model_name=LLM_MODEL_NAME, max_seq_length=LLM_MAX_SEQ_LEN,
    dtype=None, load_in_4bit=False
)
FastLanguageModel.for_inference(model_base)
resp_base = generar_itinerario(INSTRUCCION, PERFIL_TEST, model_base, tok_base)
print(resp_base)

PERFIL DE PRUEBA:
Turista de Argentina, viaja en pareja, 5 días, presupuesto medio, intereses: cultura, región: sierra, destino: Machu Picchu, temporada: seca (mayo-octubre).

✨ MODELO FINE-TUNED (PeruRoute AI):
----------------------------------------
Día 1: Llegada a Cusco. Aclimatación, mate de coca, paseo por la Plaza de Armas.
Día 2: City tour Cusco: Sacsayhuamán, Qorikancha y Mercado de San Pedro.
Día 3: Tren a Aguas Calientes. Tarde libre explorando el pueblo.
Día 4: Amanecer en Machu Picchu. Visita guiada a la ciudadela inca.
Día 5: Regreso a Cusco. Visita al Valle Sagrado: Pisac y Ollantaytambo.

Esta itineraria contiene actividades por día, mezcla de región con intereses específicos, y recomendaciones de transporte. Ajusta días, actividades según presupuesto y temporada.
==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 7.5. CUDA Too

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
¡Claro! Aquí tienes un itinerario turístico personalizado para tu viaje a Perú:

**Día 1: Lima (Sierra)**
- **Desayuno:** Restaurant local
- **Madrugada:** Visita al Museo Larco y al Templo de Hatshepsut 
- **Tarde:** Excursión a Mirador San Pedro y Parque Zoo de Lima
- **Cena:** Restaurante tradicional peruano

**Día 2: Machu Picchu (Sierra)**
- **Desayuno:** Restaurante local
- **Excursión:** Traslado a Machu Picchu, visita al sitio arqueológico, y subida al Huayna Picchu o Inti Punku
- **Cena:** Comida casera local

**Día 3: Aguas Calientes (Sierra)**
- **Desayuno:** Restaurante local
- **Madrugada:** Excursión a la Cueva de los Caquis
- **Tarde:** Visita al Puente Rupestre y al Pataz
- **Cena:** Comida casera local

**Día 4: Urubamba (Sierra)**
- **Desayuno:** Restaurante local
- **Tarde:** Excursión a Ollantaytambo
- **Cena:** Comida casera


---
# 🎨 PARTE 2 — Fine-tuning de imagen: logo PeruRoute AI en productos

Entrenamos **SDXL + DreamBooth LoRA** con 12 variaciones del logo para que el modelo aprenda a reproducir la identidad visual de **PeruRoute AI** en nuevos productos.

```
Entrada:  12 imágenes del logo en distintos fondos y condiciones
    ↓
Fine-tuning: DreamBooth LoRA — trigger word "PERUROUTE"
    ↓
Salida:   Logo aplicado en polos, tazas, gorros, cuadernos, etc.
```

## CELDA 2.1 — Preparar dataset de imágenes con captions

In [35]:
# ============================================================
# CELDA 2.1: Preparar dataset de imágenes y captions
# ============================================================
import shutil, os
from pathlib import Path
from PIL import Image

DATASET_IMG_DIR = "/kaggle/working/dataset_brand"
TARGET_SIZE     = (1024, 1024)

# Captions descriptivos para cada variación del logo
CUSTOM_CAPTIONS = {
    "01_logo_fondo_oscuro.png" : "PERUROUTE logo on a dark green background, gold andean cross symbol with circular frame, bold serif typography in dark green and gold, isolated brand identity product photo",
    "02_logo_fondo_claro.png"  : "PERUROUTE logo on a light sage green background, dark green andean cross symbol with gold diamond center, bold serif typography, clean studio product photo",
    "03_logo_fondo_negro.png"  : "PERUROUTE logo on a black background, dark green circular andean cross icon with gold accents, gold and green bold serif typography, high contrast brand identity photo",
    "04_logo_fondo_dorado.png" : "PERUROUTE logo on a gold background, dark green circular andean cross symbol, dark green bold serif typography, premium brand identity product photo",
    "05_logo_fondo_azul.png"   : "PERUROUTE logo on a navy blue background, dark green andean cross icon with gold diamond detail, white and gold bold serif typography, travel brand product photo",
    "06_logo_fondo_rojo.png"   : "PERUROUTE logo on a deep red background, dark green circular andean cross symbol with gold accents, white and gold bold serif typography, brand identity photo",
    "07_logo_invertido_horizontal.png": "PERUROUTE logo mirrored horizontally on a white background, dark green andean cross icon with gold diamond center, dark green and gold bold serif typography, flipped brand variation",
    "08_solo_icono_fondo_oscuro.png"  : "PERUROUTE brand icon isolated on white background, dark green circular andean cross symbol with gold stepped edges and gold diamond center, no text, logo mark only",
    "09_solo_icono_fondo_claro.png"   : "PERUROUTE brand icon isolated on light gray background, dark green circular andean cross with gold accents, clean icon only version, no typography",
    "10_escala_grises.png"     : "PERUROUTE logo in grayscale on white background, dark gray circular andean cross symbol with light gray accents, gray bold serif typography, monochrome brand identity",
    "11_logo_fondo_crema.png"  : "PERUROUTE logo on a cream beige background, dark green andean cross icon with gold diamond center, dark green and gold bold serif typography, warm neutral brand photo",
    "12_logo_fondo_verde_medio.png": "PERUROUTE logo on a medium forest green background, dark green circular andean cross with gold stepped pattern, white and gold bold serif typography, brand identity on brand color",
}

# Limpiar y recrear directorio
if Path(DATASET_IMG_DIR).exists():
    shutil.rmtree(DATASET_IMG_DIR)
os.makedirs(DATASET_IMG_DIR)

imagenes = sorted([f for f in Path(IMG_IMAGES_DIR).iterdir()
                   if f.suffix.lower() in ('.png','.jpg','.jpeg','.webp')])

# Eliminar archivos .txt del directorio — SD 1.5 no los soporta
import glob
for txt_file in glob.glob(f"{DATASET_IMG_DIR}/*.txt"):
    os.remove(txt_file)
    print(f"🗑️ Eliminado: {txt_file}")
print("✅ Directorio limpio — solo imágenes PNG")

print(f'📂 Imágenes encontradas: {len(imagenes)}')
if len(imagenes) == 0:
    raise FileNotFoundError(f'❌ Sube las imágenes del logo a: {IMG_IMAGES_DIR}')

for i, img_path in enumerate(imagenes):
    nombre    = img_path.name
    caption   = CUSTOM_CAPTIONS.get(nombre,
                    f"{IMG_TRIGGER_WORD} brand logo, {IMG_BRAND_COLORS}, {IMG_BRAND_STYLE}, product photo")
    base      = f"img_{i+1:03d}"

    # Redimensionar a 1024x1024
    img = Image.open(img_path).convert("RGB")
    img.thumbnail(TARGET_SIZE, Image.LANCZOS)
    canvas = Image.new("RGB", TARGET_SIZE, (255, 255, 255))
    offset = ((TARGET_SIZE[0]-img.width)//2, (TARGET_SIZE[1]-img.height)//2)
    canvas.paste(img, offset)
    canvas.save(Path(DATASET_IMG_DIR) / f"{base}.png")
    (Path(DATASET_IMG_DIR) / f"{base}.txt").write_text(caption, encoding="utf-8")

    print(f'  ✅ {i+1:02d}. {nombre:40s} → {caption[:60]}...')

print(f'\n✅ Dataset de imágenes listo: {len(imagenes)} pares imagen+caption')

✅ Directorio limpio — solo imágenes PNG
📂 Imágenes encontradas: 12
  ✅ 01. 01_logo_fondo_oscuro.png                 → PERUROUTE logo on a dark green background, gold andean cross...
  ✅ 02. 02_logo_fondo_claro.png                  → PERUROUTE logo on a light sage green background, dark green ...
  ✅ 03. 03_logo_fondo_negro.png                  → PERUROUTE logo on a black background, dark green circular an...
  ✅ 04. 04_logo_fondo_dorado.png                 → PERUROUTE logo on a gold background, dark green circular and...
  ✅ 05. 05_logo_fondo_azul.png                   → PERUROUTE logo on a navy blue background, dark green andean ...
  ✅ 06. 06_logo_fondo_rojo.png                   → PERUROUTE logo on a deep red background, dark green circular...
  ✅ 07. 07_logo_invertido_horizontal.png         → PERUROUTE logo mirrored horizontally on a white background, ...
  ✅ 08. 08_solo_icono_fondo_oscuro.png           → PERUROUTE brand icon isolated on white background, dark gree...
  ✅ 09. 09_so

## CELDA 2.2 — Descargar script de entrenamiento DreamBooth

In [33]:
# ============================================================
# CELDA 2.2: Descargar script DreamBooth LoRA SDXL
# ============================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SCRIPT_PATH = "/kaggle/working/train_dreambooth_lora.py"
if not os.path.exists(SCRIPT_PATH):
    print('📥 Descargando script DreamBooth...')
    !wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora.py \
    -O {SCRIPT_PATH}
    print('✅ Script SD 1.5 descargado.')
else:
    print('✅ Script ya disponible.')

!accelerate config default
print('✅ Accelerate configurado.')

📥 Descargando script DreamBooth...
✅ Script SD 1.5 descargado.
Configuration already exists at /root/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.
✅ Accelerate configurado.


## CELDA 2.3 — Entrenamiento DreamBooth LoRA

In [37]:
# Forzar configuración single GPU para DreamBooth
import os

config_content = """compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
"""

os.makedirs(os.path.expanduser("~/.cache/huggingface/accelerate"), exist_ok=True)
config_path = os.path.expanduser("~/.cache/huggingface/accelerate/default_config.yaml")
with open(config_path, "w") as f:
    f.write(config_content)

print(f"✅ Accelerate configurado para single GPU")
print(f"   Config guardado en: {config_path}")

✅ Accelerate configurado para single GPU
   Config guardado en: /root/.cache/huggingface/accelerate/default_config.yaml


In [29]:
import torch, gc

# Liberar todo lo que quede
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Verificar
free, total = torch.cuda.mem_get_info()
print(f"VRAM libre : {free/1024**3:.1f} GB")
print(f"VRAM total : {total/1024**3:.1f} GB")

VRAM libre : 5.9 GB
VRAM total : 14.6 GB


In [39]:
import os, glob

# Verificar y eliminar todos los .txt del directorio
txt_files = glob.glob("/kaggle/working/dataset_brand/*.txt")
print(f"Archivos .txt encontrados: {len(txt_files)}")
for f in txt_files:
    os.remove(f)
    print(f"🗑️ Eliminado: {f}")

# Verificar que solo quedan PNGs
remaining = os.listdir("/kaggle/working/dataset_brand")
print(f"\nArchivos restantes: {len(remaining)}")
for f in remaining:
    print(f"  {f}")

Archivos .txt encontrados: 12
🗑️ Eliminado: /kaggle/working/dataset_brand/img_004.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_003.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_011.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_006.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_010.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_007.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_009.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_002.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_005.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_008.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_012.txt
🗑️ Eliminado: /kaggle/working/dataset_brand/img_001.txt

Archivos restantes: 12
  img_002.png
  img_001.png
  img_005.png
  img_009.png
  img_006.png
  img_007.png
  img_004.png
  img_008.png
  img_003.png
  img_011.png
  img_010.png
  img_012.png


In [40]:
# ============================================================
# CELDA 2.3: Entrenamiento DreamBooth LoRA SDXL
# ============================================================
import subprocess

secs_per_step = 3.5
tiempo_min    = (IMG_MAX_STEPS * secs_per_step) / 60
print(f'🚀 Iniciando entrenamiento de imagen... (~{tiempo_min:.0f} min en T4)')

args = [
    "accelerate", "launch", "--num_processes=1", "--mixed_precision=fp16",
    SCRIPT_PATH,
    f"--pretrained_model_name_or_path={IMG_MODEL_NAME}",
    f"--instance_data_dir={DATASET_IMG_DIR}",
    f"--output_dir={IMG_OUTPUT_DIR}",
    f"--instance_prompt={IMG_TRIGGER_WORD} branded product with logo",
    "--resolution=512",
    "--train_batch_size=1",
    "--gradient_accumulation_steps=2",
    "--gradient_checkpointing",
    "--use_8bit_adam",
    f"--learning_rate={IMG_LR}",
    "--lr_scheduler=cosine",
    "--lr_warmup_steps=50",
    f"--max_train_steps={IMG_MAX_STEPS}",
    f"--rank={IMG_LORA_RANK}",
    "--seed=42",
    "--mixed_precision=fp16",
]

process = subprocess.Popen(args, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

if process.returncode == 0:
    print('\n✅ Entrenamiento de imagen completado!')
else:
    print(f'\n❌ Error (código: {process.returncode})')

🚀 Iniciando entrenamiento de imagen... (~12 min en T4)
W0603 00:32:09.651000 1691 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-06-03 00:32:10.179611: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780446730.204178    1691 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780446730.213961    1691 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780446730.233617    1691 computation_placer.cc:177] computation placer already registered. Pl

In [50]:
import torch, gc

del model
del tokenizer

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM libre: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

VRAM libre: 8.1 GB


In [52]:
# ============================================================
# MEJORA: Reentrenar con más steps y learning rate más bajo
# ============================================================
import torch, gc

# Liberar memoria
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM libre: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

# Eliminar .txt si quedaron
import glob, os
for f in glob.glob("/kaggle/working/dataset_brand/*.txt"):
    os.remove(f)

# Reentrenar con parámetros mejorados
import subprocess
args = [
    "accelerate", "launch", "--num_processes=1", "--mixed_precision=fp16",
    "/kaggle/working/train_dreambooth_lora.py",
    f"--pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5",
    "--instance_data_dir=/kaggle/working/dataset_brand",
    "--output_dir=/kaggle/working/brand_output_v2",
    f"--instance_prompt=PERUROUTE logo, dark green circular andean cross, gold diamond, serif typography",
    "--resolution=512",
    "--train_batch_size=1",
    "--gradient_accumulation_steps=4",
    "--gradient_checkpointing",
    "--use_8bit_adam",
    "--learning_rate=2e-5",      # más bajo que antes (era 1e-4)
    "--lr_scheduler=cosine",
    "--lr_warmup_steps=100",
    "--max_train_steps=500",     # más steps que antes (era 200)
    "--rank=32",                 # LoRA rank más alto (era 16)
    "--seed=42",
    "--mixed_precision=fp16",
]

process = subprocess.Popen(args, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

if process.returncode == 0:
    print('\n✅ Reentrenamiento completado!')
else:
    print(f'\n❌ Error (código: {process.returncode})')

VRAM libre: 8.1 GB
W0603 01:20:06.168000 1970 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-06-03 01:20:06.666341: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780449606.691004    1970 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780449606.700012    1970 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780449606.720446    1970 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking

## CELDA 2.4 — Cargar SDXL con LoRA y generar mockups

In [53]:
# ============================================================
# CELDA 2.4: Cargar SDXL + LoRA y generar mockups del logo
# ============================================================
import torch
import math
from pathlib import Path
from PIL import Image
from IPython.display import display
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
from diffusers import StableDiffusionPipeline

print('📥 Cargando SDXL base... (~2-3 min)')


pipe = StableDiffusionPipeline.from_pretrained(
    IMG_MODEL_NAME, torch_dtype=torch.float16,
    safety_checker=None,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True)
pipe.enable_xformers_memory_efficient_attention()
pipe.enable_model_cpu_offload()
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True)
pipe.enable_xformers_memory_efficient_attention()
pipe.enable_model_cpu_offload()

print(f'🔌 Cargando LoRA {IMG_TRIGGER_WORD}...')
pipe.load_lora_weights("/kaggle/working/brand_output_v2")

# Productos a generar
PROMPTS_MOCKUP = [
    f"{IMG_TRIGGER_WORD} logo on a white crew neck t-shirt, flat lay, studio photography",
    f"{IMG_TRIGGER_WORD} logo embroidered on a navy blue cap, side view, product photo",
    f"{IMG_TRIGGER_WORD} logo on a ceramic coffee mug, {IMG_BRAND_COLORS}, white background",
    f"{IMG_TRIGGER_WORD} logo on a hardcover notebook, {IMG_BRAND_COLORS}, flat lay",
    f"{IMG_TRIGGER_WORD} logo on a stainless steel water bottle, {IMG_BRAND_COLORS}, product photo",
    f"{IMG_TRIGGER_WORD} logo on a canvas tote bag, natural fabric, studio photography",
]
NEGATIVE = "blurry, low quality, distorted logo, deformed text, watermark, ugly"

mockups_dir = Path(IMG_OUTPUT_DIR) / "mockups"
mockups_dir.mkdir(exist_ok=True)

print(f'\n🎨 Generando {len(PROMPTS_MOCKUP)} mockups...')
imgs_gen = []
for i, prompt in enumerate(PROMPTS_MOCKUP):
    print(f'  [{i+1}/{len(PROMPTS_MOCKUP)}] {prompt[:65]}...')
    img = pipe(
        prompt=prompt, negative_prompt=NEGATIVE,
        num_inference_steps=30, guidance_scale=7.5,
        width=1024, height=1024,
        generator=torch.Generator().manual_seed(42+i)
    ).images[0]
    img.save(mockups_dir / f"mockup_{i+1:02d}.png")
    imgs_gen.append(img)

# Galería
cols    = 3
rows    = math.ceil(len(imgs_gen) / cols)
thumb   = (512, 512)
galeria = Image.new("RGB", (cols*thumb[0], rows*thumb[1]), (240,240,240))
for idx, img in enumerate(imgs_gen):
    t = img.copy(); t.thumbnail(thumb)
    galeria.paste(t, ((idx%cols)*thumb[0], (idx//cols)*thumb[1]))
galeria.save(mockups_dir / "galeria_mockups.png")

print(f'\n✅ {len(imgs_gen)} mockups generados.')
display(galeria)

📥 Cargando SDXL base... (~2-3 min)


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

🔌 Cargando LoRA PERUROUTE...

🎨 Generando 6 mockups...
  [1/6] PERUROUTE logo on a white crew neck t-shirt, flat lay, studio pho...


  0%|          | 0/30 [00:00<?, ?it/s]

  [2/6] PERUROUTE logo embroidered on a navy blue cap, side view, product...


...

  0%|          | 0/30 [00:00<?, ?it/s]

  [6/6] PERUROUTE logo on a canvas tote bag, natural fabric, studio photo...


  0%|          | 0/30 [00:00<?, ?it/s]


✅ 6 mockups generados.


<PIL.Image.Image image mode=RGB size=1536x1024>

Por limitaciones de VRAM en T4 usamos SD 1.5 en lugar de SDXL con 200 steps. Los resultados muestran el pipeline funcionando; con SDXL y 500+ steps la fidelidad del logo mejoraría significativamente

## CELDA 2.5 — Comparativa: modelo base vs fine-tuned (imagen)

In [43]:
# ============================================================
# CELDA 2.5: Comparativa visual antes/después del LoRA
# ============================================================
from PIL import ImageDraw

PROMPT_TEST = f"{IMG_TRIGGER_WORD} logo on a white tote bag, flat lay, studio photography"

print('📷 Generando SIN LoRA (modelo base)...')
pipe.unload_lora_weights()
img_base = pipe(
    prompt=PROMPT_TEST, num_inference_steps=30, guidance_scale=7.5,
    width=1024, height=1024, generator=torch.Generator().manual_seed(42)
).images[0]

print(f'🔌 Generando CON LoRA ({IMG_TRIGGER_WORD})...')
pipe.load_lora_weights(IMG_OUTPUT_DIR)
img_lora = pipe(
    prompt=PROMPT_TEST, num_inference_steps=30, guidance_scale=7.5,
    width=1024, height=1024, generator=torch.Generator().manual_seed(42)
).images[0]

# Composición comparativa
thumb = (512, 512)
comp  = Image.new("RGB", (1064, 592), (255,255,255))
b = img_base.copy(); b.thumbnail(thumb)
l = img_lora.copy(); l.thumbnail(thumb)
comp.paste(b, (20, 60))
comp.paste(l, (552, 60))
draw = ImageDraw.Draw(comp)
draw.rectangle([0,0,1064,55], fill=(245,245,245))
draw.text((20,  15), "❌ SIN LoRA — modelo genérico",            fill=(180,50,50))
draw.text((552, 15), f"✅ CON LoRA — {IMG_TRIGGER_WORD} brand",  fill=(50,140,80))
draw.text((20, 575), f"Prompt: {PROMPT_TEST[:90]}",               fill=(120,120,120))

comp.save(Path(IMG_OUTPUT_DIR) / "comparacion_imagen.png")
print('\n📊 Comparativa antes/después:')
display(comp)

📷 Generando SIN LoRA (modelo base)...


  0%|          | 0/30 [00:00<?, ?it/s]

🔌 Generando CON LoRA (PERUROUTE)...


  0%|          | 0/30 [00:00<?, ?it/s]


📊 Comparativa antes/después:


<PIL.Image.Image image mode=RGB size=1064x592>

---
# 🚀 PARTE 3 — Pipeline integrado: perfil del turista → itinerario + imagen

Aquí conectamos los dos módulos: el LLM genera el itinerario y extrae el destino principal, que luego se usa como prompt para generar la imagen del destino con el logo PeruRoute AI.

In [45]:
# Recargar modelo LLM fine-tuned para el pipeline integrado
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = LLM_OUTPUT_DIR,
    max_seq_length= LLM_MAX_SEQ_LEN,
    dtype         = None,
    load_in_4bit  = False,
)
print("✅ Modelo LLM fine-tuned recargado")

==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Modelo LLM fine-tuned recargado


In [46]:
# ============================================================
# CELDA 3.1: Pipeline completo — perfil → itinerario → imagen
# ============================================================
import re
from IPython.display import display, HTML

FastLanguageModel.for_inference(model)

DESTINO_PROMPTS = {
    "machu picchu" : "ancient inca citadel Machu Picchu Peru, misty mountains, dramatic landscape",
    "cusco"        : "Cusco Peru colonial plaza, andean mountains, colorful streets",
    "lago titicaca": "Lake Titicaca Peru, floating islands, blue waters, andean sky",
    "iquitos"      : "Amazon rainforest Iquitos Peru, tropical jungle, river wildlife",
    "mancora"      : "Mancora Peru beach, Pacific Ocean, surf, golden sand",
    "lima"         : "Lima Peru Miraflores coast, modern city, Pacific Ocean cliffs",
    "paracas"      : "Paracas Peru desert coast, Ballestas Islands, sea lions, birds",
    "huaraz"       : "Huaraz Peru Cordillera Blanca, snow peaks, trekking landscape",
}

def extraer_destino(texto):
    """Detecta el destino principal mencionado en el itinerario."""
    texto_lower = texto.lower()
    for dest in DESTINO_PROMPTS:
        if dest in texto_lower:
            return dest
    return "machu picchu"  # default

def pipeline_completo(perfil):
    """Ejecuta el pipeline completo para un perfil de turista."""

    print('\n' + '='*65)
    print('📋 PERFIL DEL TURISTA:')
    print(f'   {perfil}')
    print('='*65)

    # ── PASO 1: Generar itinerario con LLM fine-tuned ──
    print('\n🤖 Paso 1: Generando itinerario con LLM fine-tuned...')
    inst      = "Genera un itinerario turístico personalizado para el Perú basado en el perfil del viajero."
    itinerario = generar_itinerario(inst, perfil, model, tokenizer)
    print('\n📅 ITINERARIO GENERADO:')
    print(itinerario)

    # ── PASO 2: Extraer destino y construir prompt imagen ──
    print('\n🖼️  Paso 2: Generando imagen del destino...')
    destino      = extraer_destino(itinerario)
    desc_destino = DESTINO_PROMPTS[destino]
    prompt_img   = (
        f"{IMG_TRIGGER_WORD} travel brand promotional photo, "
        f"{desc_destino}, "
        f"professional travel photography, vibrant colors, golden hour"
    )
    print(f'   Destino detectado : {destino.title()}')
    print(f'   Prompt imagen     : {prompt_img[:80]}...')

    # ── PASO 3: Generar imagen con SDXL + LoRA ──
    pipe.load_lora_weights(IMG_OUTPUT_DIR)
    imagen = pipe(
        prompt          = prompt_img,
        negative_prompt = "blurry, low quality, ugly, watermark",
        num_inference_steps=30, guidance_scale=7.5,
        width=1024, height=1024,
        generator=torch.Generator().manual_seed(42)
    ).images[0]

    print('\n🏔️  IMAGEN DEL DESTINO:')
    display(imagen)

    return itinerario, imagen

# ── Ejemplo 1 ──
perfil_1 = "Turista de España, viaja en pareja, 7 días, presupuesto alto, intereses: cultura y gastronomía, destino: Machu Picchu, temporada: seca."
itin_1, img_1 = pipeline_completo(perfil_1)

# ── Ejemplo 2 ──
perfil_2 = "Turista de Brasil, viaja solo, 5 días, presupuesto medio, intereses: naturaleza y aventura, destino: Iquitos, temporada: lluvia."
itin_2, img_2 = pipeline_completo(perfil_2)


📋 PERFIL DEL TURISTA:
   Turista de España, viaja en pareja, 7 días, presupuesto alto, intereses: cultura y gastronomía, destino: Machu Picchu, temporada: seca.

🤖 Paso 1: Generando itinerario con LLM fine-tuned...

📅 ITINERARIO GENERADO:
Día 1: Llegada a Cusco. Aclimatación, mate de coca, paseo por la Plaza de Armas.
Día 2: City tour Cusco: Sacsayhuamán, Qorikancha y Mercado de San Pedro.
Día 3: Tren a Aguas Calientes. Tarde libre explorando el pueblo.
Día 4: Amanecer en Machu Picchu. Visita guiada a la ciudadela inca.
Día 5: Regreso a Cusco. Visita al Valle Sagrado: Pisac y Ollantaytambo.
Día 6: Día libre en Cusco o excursión a la Montaña Arcoíris.
Día 7: Vuelo de regreso desde Cusco.

Nota de temporada: La seca (mayo-octubre) es la mejor época para trekking y fotografía. Reserva con anticipación.
Índice de dificultad: bajo. Índice de confort: alto. 🌿🔥✨ ¡Vuela alto!, peruano turista.

🖼️  Paso 2: Generando imagen del destino...
   Destino detectado : Machu Picchu
   Prompt imagen   

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


  0%|          | 0/30 [00:00<?, ?it/s]


🏔️  IMAGEN DEL DESTINO:


<PIL.Image.Image image mode=RGB size=1024x1024>


📋 PERFIL DEL TURISTA:
   Turista de Brasil, viaja solo, 5 días, presupuesto medio, intereses: naturaleza y aventura, destino: Iquitos, temporada: lluvia.

🤖 Paso 1: Generando itinerario con LLM fine-tuned...

📅 ITINERARIO GENERADO:
Día 1: Llegada a Iquitos. Paseo por el Malecón Tarapoto.
Día 2: Reserva Nacional Pacaya-Samiria: avistamiento de fauna.
Día 3: Canopy y kayak por tributarios del Amazonas.
Día 4: Visita a comunidades nativas. Medicina tradicional.
Día 5: Mercado de Belén. Vuelo de regreso.

Esta itineraria combina visita a puntos turísticos importantes, actividad por día, y outfit recomendado para each day. Asegura experiencias de naturaleza, contacto con cultura local y vista panorámica de la ciudad. ¿Hay something else? ¡Crea el itinerario para el Perú! 🇰🇷 🔗 ¡Reserva tu tour aquí! 💵

🖼️  Paso 2: Generando imagen del destino...
   Destino detectado : Iquitos
   Prompt imagen     : PERUROUTE travel brand promotional photo, Amazon rainforest Iquitos Peru, tropic...


  0%|          | 0/30 [00:00<?, ?it/s]


🏔️  IMAGEN DEL DESTINO:


<PIL.Image.Image image mode=RGB size=1024x1024>

---
# 📊 PARTE 4 — Resumen de métricas y comparativas

Comparamos cualitativamente el modelo base vs fine-tuned en ambos módulos.

In [47]:
# ============================================================
# CELDA 4.1: Resumen de resultados del proyecto
# ============================================================
print('=' * 65)
print('📊 RESUMEN DE RESULTADOS — PeruRoute AI')
print('=' * 65)

print(f"""
MÓDULO 1 — Fine-tuning LLM
  Modelo base    : {LLM_MODEL_NAME}
  Técnica        : QLoRA 4-bit (r={LLM_LORA_R}, alpha={LLM_LORA_ALPHA})
  Dataset        : 800 ejemplos sintéticos (formato Alpaca)
  Épocas         : {LLM_EPOCHS}
  Loss final     : ver Celda 1.4
  Mejora clave   : Itinerarios estructurados día a día con
                   tips locales vs respuestas genéricas del modelo base

MÓDULO 2 — Fine-tuning Imagen
  Modelo base    : SDXL 1.0
  Técnica        : DreamBooth LoRA (rank={IMG_LORA_RANK})
  Dataset        : 12 variaciones del logo PeruRoute AI
  Pasos          : {IMG_MAX_STEPS}
  Trigger word   : {IMG_TRIGGER_WORD}
  Mejora clave   : Logo reconocible en nuevos productos vs
                   diseño genérico sin identidad de marca

PIPELINE INTEGRADO
  Flujo          : Perfil → LLM → itinerario → destino → SDXL → imagen
  Casos probados : 2 perfiles distintos (España/pareja, Brasil/solo)
""")

print('\n🎯 Estimación de impacto:')
print('  • 80% reducción en tiempo de planificación de itinerarios')
print('  • 5x más consultas atendidas por día por agencia')
print('  • Eliminación de sesiones fotográficas para mockups de marca')

📊 RESUMEN DE RESULTADOS — PeruRoute AI

MÓDULO 1 — Fine-tuning LLM
  Modelo base    : unsloth/Qwen2.5-3B-Instruct
  Técnica        : QLoRA 4-bit (r=16, alpha=16)
  Dataset        : 800 ejemplos sintéticos (formato Alpaca)
  Épocas         : 3
  Loss final     : ver Celda 1.4
  Mejora clave   : Itinerarios estructurados día a día con
                   tips locales vs respuestas genéricas del modelo base

MÓDULO 2 — Fine-tuning Imagen
  Modelo base    : SDXL 1.0
  Técnica        : DreamBooth LoRA (rank=16)
  Dataset        : 12 variaciones del logo PeruRoute AI
  Pasos          : 200
  Trigger word   : PERUROUTE
  Mejora clave   : Logo reconocible en nuevos productos vs
                   diseño genérico sin identidad de marca

PIPELINE INTEGRADO
  Flujo          : Perfil → LLM → itinerario → destino → SDXL → imagen
  Casos probados : 2 perfiles distintos (España/pareja, Brasil/solo)


🎯 Estimación de impacto:
  • 80% reducción en tiempo de planificación de itinerarios
  • 5x más consul

---
# 💾 CELDA FINAL — Guardar todos los artefactos

Guarda en Drive los pesos, mockups y resultados del pipeline.

In [48]:
# ============================================================
# CELDA FINAL: Guardar todos los artefactos del proyecto
# ============================================================
from pathlib import Path

# Guardar imágenes del pipeline integrado
pipeline_dir = Path("/kaggle/working/peruroute_outputs")
pipeline_dir.mkdir(exist_ok=True)

img_1.save(pipeline_dir / "pipeline_ejemplo1_machu_picchu.png")
img_2.save(pipeline_dir / "pipeline_ejemplo2_iquitos.png")

# Guardar itinerarios como texto
(pipeline_dir / "itinerario_ejemplo1.txt").write_text(itin_1, encoding="utf-8")
(pipeline_dir / "itinerario_ejemplo2.txt").write_text(itin_2, encoding="utf-8")

# Verificar archivos LoRA imagen
loras = list(Path(IMG_OUTPUT_DIR).glob("*.safetensors"))

print('📁 Artefactos guardados en Kaggle Working:')
print(f'   ✅ LLM LoRA          → {LLM_OUTPUT_DIR}')
print(f'   ✅ SDXL LoRA         → {IMG_OUTPUT_DIR}')
print(f'   ✅ Mockups generados → {IMG_OUTPUT_DIR}/mockups/')
print(f'   ✅ Pipeline outputs  → {pipeline_dir}')
print(f'\n   LoRA imagen: {loras[-1].name if loras else "ver carpeta output"}' )
print('\n🎉 ¡Proyecto PeruRoute AI completo!')

📁 Artefactos guardados en Google Drive:
   ✅ LLM LoRA          → /kaggle/working/peruroute_llm_lora
   ✅ SDXL LoRA         → /kaggle/working/brand_output
   ✅ Mockups generados → /kaggle/working/brand_output/mockups/
   ✅ Pipeline outputs  → /kaggle/working/peruroute_outputs

   LoRA imagen: pytorch_lora_weights.safetensors

🎉 ¡Proyecto PeruRoute AI completo!
